# Image Warping

**Image warping** applies a geometric transformation to an image: it moves each
pixel to a new location according to a mapping between the **source** image and
the **destination** image. Affine transforms, perspective (homography)
transforms, and lens-distortion correction are all instances of warping.

The central practical question is: *in which direction do we scan?*

## Forward vs. Inverse Warping

Let $T$ map a source coordinate $(x, y)$ to a destination coordinate
$(x', y') = T(x, y)$.

### Forward warping (push)

Iterate over every **source** pixel, transform its coordinate, and write its
value to the destination:

$$
(x', y') = T(x, y), \qquad dst(x', y') = src(x, y)
$$

This is intuitive but has two problems:

* **Holes.** The transformed coordinates $(x', y')$ are generally
  **non-integer**, and some destination pixels are never hit by any source
  pixel. The result is peppered with gaps (especially where the image is
  stretched).
* **Collisions / overlap.** Several source pixels can map to the same
  destination pixel (where the image is compressed), so you must decide how to
  combine them.

### Inverse warping (pull) — what OpenCV actually does

Iterate over every **destination** pixel, apply the **inverse** transform to
find where it came from in the source, and **sample** that source location:

$$
(x, y) = T^{-1}(x', y'), \qquad dst(x', y') = src(x, y)
$$

Because we loop over the destination grid, **every output pixel is assigned
exactly one value — there are no holes and no collisions.** The recovered
source coordinate $(x, y)$ is again non-integer, but now that is easy to handle:
we simply **interpolate** between the surrounding source pixels.

> This is why warping is implemented as *sample the destination, look up the
> source*. OpenCV's `warpAffine`, `warpPerspective`, and `remap` all use inverse
> (pull) mapping.

Note on the transform matrix: by default `warpAffine` / `warpPerspective` treat
the supplied matrix `M` as the **forward** (source → destination) transform and
invert it internally. If you already have the destination → source mapping, pass
the flag `cv::WARP_INVERSE_MAP` and OpenCV will use `M` directly without
inverting it.


## Interpolation

Inverse warping lands on a **fractional** source coordinate
$(x, y)$, so we must estimate the intensity between integer pixel centers.
Let $x_0 = \lfloor x \rfloor$, $y_0 = \lfloor y \rfloor$,
$a = x - x_0$, and $b = y - y_0$.

| Method | OpenCV flag | Neighborhood | Cost | Quality |
| ------ | ----------- | ------------ | ---- | ------- |
| **Nearest neighbor** | `INTER_NEAREST` | 1 pixel | cheapest | blocky / aliased; preserves label values (use for masks) |
| **Bilinear** | `INTER_LINEAR` (default) | 2×2 = 4 pixels | low | smooth; good general default |
| **Bicubic** | `INTER_CUBIC` | 4×4 = 16 pixels | higher | sharper edges, can slightly overshoot (ringing) |

Two more are useful in practice: `INTER_AREA` (best for **shrinking**, does
area-averaging) and `INTER_LANCZOS4` (8×8, highest quality / slowest).

**Nearest neighbor** just rounds to the closest pixel:

$$
dst = src\big(\,\text{round}(y),\ \text{round}(x)\,\big)
$$

**Bilinear** takes the weighted average of the four surrounding pixels:

$$
dst = (1-a)(1-b)\,I_{00} + a(1-b)\,I_{10} + (1-a)b\,I_{01} + ab\,I_{11}
$$

where $I_{00}=src(y_0,x_0)$, $I_{10}=src(y_0,x_0{+}1)$,
$I_{01}=src(y_0{+}1,x_0)$, $I_{11}=src(y_0{+}1,x_0{+}1)$.

**Bicubic** fits a cubic polynomial over the 4×4 neighborhood, giving smoother
gradients and crisper edges at the price of more computation.


## OpenCV Warping Functions

![affine and perspective transformation matrices](images/affine_perspective_transformation_matrix.svg)

### `cv::warpAffine(src, dst, M, dsize)`

* `M` is a **2×3** matrix. An affine transform is a linear map plus a
  translation, so it maps

$$
\begin{bmatrix} x' \\ y' \end{bmatrix} =
M \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

* Preserves **straight lines and parallelism** (parallel lines stay parallel);
  it covers translation, rotation, scaling, and shear.
* Build `M` with `cv::getAffineTransform` (from **3** point correspondences),
  `cv::getRotationMatrix2D`, or by hand.

### `cv::warpPerspective(src, dst, M, dsize)`

* `M` is a **3×3** homography. Points are mapped in **homogeneous**
  coordinates and divided by the third component:

$$
\begin{bmatrix} x'w \\ y'w \\ w \end{bmatrix} =
M \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
x' = \frac{x'w}{w},\quad y' = \frac{y'w}{w}
$$

* Preserves **straight lines** but **not parallelism** — this is what produces
  the "keystone" perspective effect.
* Build `M` with `cv::getPerspectiveTransform` (from **4** correspondences) or
  `cv::findHomography`.

### `cv::remap(src, dst, map1, map2, interpolation)`

* The **general** per-pixel warp. For each destination pixel it reads the
  source coordinate straight out of the lookup tables you provide:

$$
dst(y, x) = src\big(map_y(y, x),\ map_x(y, x)\big)
$$

* The mapping is **arbitrary** and non-parametric — it need not correspond to
  any matrix — which is exactly what you need for **lens-distortion
  correction**, **stereo rectification**, or any hand-built distortion field.

## Geometric Warp vs. Pixel Remap

`warpAffine` and `warpPerspective` are **parametric geometric warps**: you give
a small matrix, and OpenCV computes each source coordinate on the fly (via the
inverse transform) and interpolates.

`remap` is the underlying **pixel-remap primitive**: you hand it explicit
per-pixel `map_x` / `map_y` tables and it does the sampling and interpolation.

They are two views of the same "pull" operation. In fact `warpAffine` /
`warpPerspective` can be seen as `remap` where the maps are generated from the
matrix. Use the matrix functions when the transform is a clean affine or
homography; drop down to `remap` when the mapping is irregular (e.g.
`cv::initUndistortRectifyMap` precomputes the maps that `remap` then applies to
undistort every frame cheaply).


## Demo

The snippet below builds a checkerboard grid and applies both an affine and a
perspective warp. `getAffineTransform` needs 3 point correspondences;
`getPerspectiveTransform` needs 4.

> The visualization uses `cv2.imshow`, which opens native GUI windows. Run it as
> a script (not headless); press any key to close the windows.


In [ ]:
import cv2
import numpy as np

# Generate a simple checkerboard image
def create_grid(size=400, step=40):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    for y in range(0, size, step):
        for x in range(0, size, step):
            if (x // step + y // step) % 2 == 0:
                img[y:y+step, x:x+step] = (255, 255, 255)
    return img

# Create base grid
img = create_grid()

rows, cols, _ = img.shape

# Define 3 source and destination points for affine transform
pts1_affine = np.float32([[50, 50], [200, 50], [50, 200]])
pts2_affine = np.float32([[10, 100], [200, 50], [100, 250]])

# Compute affine transform matrix
M_affine = cv2.getAffineTransform(pts1_affine, pts2_affine)
warp_affine = cv2.warpAffine(img, M_affine, (cols, rows))

# Define 4 points for perspective transform
pts1_persp = np.float32([[50, 50], [350, 50], [50, 350], [350, 350]])
pts2_persp = np.float32([[10, 100], [380, 50], [80, 350], [330, 380]])

# Compute perspective transform (3x3)
H = cv2.getPerspectiveTransform(pts1_persp, pts2_persp)
warp_persp = cv2.warpPerspective(img, H, (cols, rows))

# Visualize all
cv2.imshow('Original Grid', img)
cv2.imshow('Affine Warp', warp_affine)
cv2.imshow('Perspective Warp', warp_persp)
cv2.waitKey(0)
cv2.destroyAllWindows()

